# Resumo Unico para Teste (AC 25/26)

Material consolidado de TP1-TP6, sem repeticoes e com **EDA minima**.

Objetivo: servir como folha de consulta rapida para implementar pipelines corretos e responder perguntas teoricas.

## 0) Mapa do que realmente importa

1. Workflow correto: split -> CV no treino -> tuning -> threshold (se necessario) -> teste final uma unica vez
2. Preprocessamento sem data leakage (Pipeline/ColumnTransformer)
3. Modelos principais: Logistic, KNN, Naive Bayes, Trees, SVM, Gradient Boosting/Random Forest
4. Metricas: Accuracy, Balanced Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC
5. NLP: cleaning + BoW/TF-IDF + features manuais + SelectKBest(chi2)
6. Unsupervised: KMeans/Agglomerative + PCA (variancia explicada, loadings)
7. Imbalance e interpretacao: SMOTE/undersampling, calibration, permutation importance
8. Diferenca NumPy vs pandas no sklearn (ordem/nomes de colunas)

## 1) Workflow padrao (evitar erros metodologicos)

- **Nunca** usar o conjunto de teste para escolher modelo/hiperparametros/threshold
- Fazer tudo no treino com CV
- Teste apenas no fim, uma vez

Fluxo recomendado:
- `train_test_split(..., stratify=y, random_state=42)`
- Definir pipeline
- `GridSearchCV` (ou outro search) no treino
- Opcional: threshold tuning com previsoes out-of-fold
- Fit final no treino inteiro
- Avaliacao final no teste

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, roc_auc_score, average_precision_score

# Template geral (adaptar colunas/modelo/param_grid ao dataset)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

num_cols = X.select_dtypes(include=["number"]).columns
cat_cols = X.select_dtypes(exclude=["number"]).columns

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

pipe = Pipeline([
    ("prep", preprocess),
    ("model", model)
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(pipe, param_grid=param_grid, cv=cv, scoring="f1", n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("Best params:", grid.best_params_)
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("F1:", round(f1_score(y_test, y_pred, average="binary"), 4))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

## 2) Metricas: quando usar

- `Accuracy`: classes equilibradas e custo de erro semelhante
- `Balanced Accuracy`: classes desbalanceadas
- `Precision`: quando falso positivo custa muito
- `Recall`: quando falso negativo custa muito
- `F1`: equilibrio entre precision e recall
- `ROC-AUC`: comparacao global entre thresholds
- `PR-AUC`: melhor quando classe positiva e rara

Formulas:
- Precision = TP / (TP + FP)
- Recall = TP / (TP + FN)
- F1 = 2 * (Precision * Recall) / (Precision + Recall)

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

# Se o modelo tiver predict_proba
scores = best_model.predict_proba(X_test)[:, 1]
print("ROC-AUC:", round(roc_auc_score(y_test, scores), 4))
print("PR-AUC (AP):", round(average_precision_score(y_test, scores), 4))

fpr, tpr, _ = roc_curve(y_test, scores)
precision, recall, _ = precision_recall_curve(y_test, scores)

## 3) Threshold tuning (tema forte de TP5)

Ideia correta:
- Escolher threshold com dados de treino via previsoes out-of-fold
- Nao ajustar threshold no teste

Exemplo: escolher threshold que maximiza F1 no treino.

In [ ]:
# OOF scores no treino
oof_scores = cross_val_predict(
    best_model, X_train, y_train, cv=cv, method="predict_proba", n_jobs=-1
)[:, 1]

thresholds = np.linspace(0.1, 0.9, 81)
f1s = []
for thr in thresholds:
    pred_thr = (oof_scores >= thr).astype(int)
    f1s.append(f1_score(y_train, pred_thr))

best_thr = thresholds[np.argmax(f1s)]
print("Best threshold (train OOF):", round(best_thr, 3))

# Avaliacao no teste com threshold escolhido no treino
test_scores = best_model.predict_proba(X_test)[:, 1]
y_test_thr = (test_scores >= best_thr).astype(int)
print("F1 no teste (threshold tuned):", round(f1_score(y_test, y_test_thr), 4))

## 4) Modelos supervisionados (resumo rapido)

- **Logistic Regression**: baseline forte em texto/tabular linear
- **KNN**: simples, sensivel a escala e ruido
- **Naive Bayes (Multinomial/Gaussian)**: muito forte em texto BoW
- **Decision Tree**: interpretavel, pode overfit
- **SVM (RBF)**: bom em fronteiras nao lineares
- **Gradient Boosting / Random Forest**: bom desempenho em tabular

SVM intuicao:
- `C` alto: penaliza erros, fronteira mais rigida
- `gamma` alto: fronteira mais local/complexa (risco overfit)

## 5) Unsupervised: KMeans, Agglomerative, PCA

KMeans:
- escolher `k` com Elbow (inercia) + Silhouette

Agglomerative:
- clustering hierarquico, alternativa ao KMeans

PCA:
- reduzir dimensao mantendo variancia
- usar `explained_variance_ratio_` e variancia acumulada
- loadings mostram contribuicao das features para cada PC

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

Xs = StandardScaler().fit_transform(X_num)

# KMeans
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_km = kmeans.fit_predict(Xs)
print("Silhouette KMeans:", round(silhouette_score(Xs, labels_km), 4))

# Agglomerative
agg = AgglomerativeClustering(n_clusters=3)
labels_ag = agg.fit_predict(Xs)
print("Silhouette Agglomerative:", round(silhouette_score(Xs, labels_ag), 4))

# PCA
pca = PCA()
Z = pca.fit_transform(Xs)
print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Cumulative variance:", np.cumsum(pca.explained_variance_ratio_))

## 6) NLP pipeline (Spam + Mini Challenge)

Passos essenciais:
1. Limpeza simples de texto
2. Vetorizacao (`CountVectorizer` ou `TfidfVectorizer`)
3. (Opcional) features manuais
4. (Opcional) `SelectKBest(chi2)`
5. CV estratificada
6. Comparar modelos (MNB vs KNN/Logistic)

In [ ]:
import re
import string
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.naive_bayes import MultinomialNB

def clean_text(s):
    s = s.lower()
    s = re.sub(r"\s+", " ", s).strip()
    s = s.translate(str.maketrans("", "", string.punctuation))
    s = re.sub(r"\d+", "", s)
    return s

Xtr_clean = X_train_text.apply(clean_text)
Xte_clean = X_test_text.apply(clean_text)

vec = CountVectorizer()
Xtr_bow = vec.fit_transform(Xtr_clean)
Xte_bow = vec.transform(Xte_clean)

k = int(0.75 * Xtr_bow.shape[1])
selector = SelectKBest(chi2, k=k)
Xtr_sel = selector.fit_transform(Xtr_bow, y_train)
Xte_sel = selector.transform(Xte_bow)

clf = MultinomialNB()
clf.fit(Xtr_sel, y_train)
pred = clf.predict(Xte_sel)
print(classification_report(y_test, pred))

## 7) Comparacao estatistica de modelos (Paired t-test)

Se tiveres scores fold-a-fold dos dois modelos no mesmo CV:
- H0: medias iguais
- p < 0.05: diferenca estatisticamente significativa

In [ ]:
from scipy.stats import ttest_rel

# Ex.: lista de F1 por fold
# scores_a = [...]
# scores_b = [...]
t_stat, p_value = ttest_rel(scores_a, scores_b)
print("p-value:", round(p_value, 6))

## 8) Imbalanced learning + calibracao + interpretacao

Imbalance:
- usar `balanced_accuracy`, `F1`, PR-AUC
- aplicar SMOTE/undersampling **dentro de pipeline**

Interpretacao:
- feature importance de arvore pode ser enviesada
- permutation importance e mais robusta

Calibracao:
- importante quando probabilidades sao usadas para decisao (threshold/custo)

In [ ]:
from sklearn.metrics import balanced_accuracy_score
from sklearn.inspection import permutation_importance
from sklearn.calibration import CalibratedClassifierCV

# balanced accuracy
print("Balanced Acc:", round(balanced_accuracy_score(y_test, y_pred), 4))

# calibracao (exemplo)
cal_model = CalibratedClassifierCV(best_model, cv=5, method="sigmoid")
cal_model.fit(X_train, y_train)

# permutation importance (exemplo)
perm = permutation_importance(best_model, X_test, y_test, n_repeats=10, random_state=42)
print("Permutation importance calculada.")

## 9) NumPy vs pandas no sklearn (TP6 importante)

- Com pandas, transformers guardam nomes de colunas e ajudam a detetar inconsistencias
- Com NumPy, so existe posicao de coluna (mais risco de erro silencioso)
- Regra pratica: manter DataFrame ate ao fim do preprocessamento

## 10) EDA minima (para nao gastar folhas)

Checklist curto:
- `shape`, `dtypes`, `isna().sum()`
- distribuicao da target (`value_counts(normalize=True)`)
- 1 grafico rapido (histograma ou boxplot) so para detetar outliers absurdos

Depois disso: passar logo para pipeline/modelacao.

## 11) Perguntas teoricas que costumam sair

1. Porque separar treino e teste no inicio?
2. Porque CV apenas no treino?
3. O que e data leakage?
4. Diferenca entre overfitting e underfitting?
5. Efeito de `C` e `gamma` no SVM?
6. Quando usar ROC-AUC vs PR-AUC?
7. Porque ajustar threshold no treino (OOF) e nao no teste?
8. Porque scaler dentro de pipeline?
9. O que representa cada principal component no PCA?
10. KMeans: como escolher `k` com inercia e silhouette?

## 12) Mini-template de resposta em teste (texto curto)

- Dividi dados em treino/teste com estratificacao.
- Fiz preprocessamento num pipeline para evitar leakage.
- Comparei modelos por CV no treino e escolhi por metrica adequada ao problema.
- Ajustei hiperparametros com GridSearchCV.
- Quando necessario, ajustei threshold com previsoes OOF no treino.
- Avaliei no teste apenas no fim com confusion matrix + metricas principais.